# MineToday V136 — Generator-Conditioned Ordinal Whole-List Fusion (GC-OWLF)

**Frozen experiment contract.** This notebook implements the agreed post-V135 V136 architecture only. It is a clean organizer-data whole-list experiment: 17 user×module candidates, exact-value control (C0), ordinal LambdaRank without generator state (C1), and ordinal LambdaRank with generator state (C2). Pseudo-GT, public-score inversion, named test-user rules, previous submission predictions, and UID/hash shortcuts are excluded from model construction.

The operational benchmark is public V132 = `0.66219`, but V132 predictions are never used as model features/labels. V136 earns a submission candidate only through the pre-registered gates encoded at the end of this notebook. Otherwise the notebook emits a HOLD decision.

In [1]:
# =========================
# 0. Runtime, dependencies, deterministic configuration
# =========================
import os, sys, re, gc, json, ast, math, time, hashlib, random, warnings, subprocess, platform
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('PYTHONHASHSEED','20260909')
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
os.environ.setdefault('MKL_NUM_THREADS','4')
os.environ.setdefault('NUMEXPR_NUM_THREADS','4')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY','1')
warnings.filterwarnings('ignore')

SEED=20260909
FOLD_SEED=20260907
random.seed(SEED)

# Install only missing/incompatible packages. Kaggle internet must be enabled only when models are not attached as datasets.
def ensure_runtime():
    import importlib.metadata as im
    need=[]
    try: im.version('lightgbm')
    except Exception: need.append('lightgbm==4.6.0')
    try:
        tv=im.version('transformers')
        if int(tv.split('.')[0])>=5: need += ['transformers==4.57.1','sentence-transformers==5.1.1']
    except Exception: need += ['transformers==4.57.1','sentence-transformers==5.1.1']
    if need:
        subprocess.check_call([sys.executable,'-m','pip','install','-q']+need)
ensure_runtime()

import numpy as np, pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import lightgbm as lgb

np.random.seed(SEED)
OUT=Path('/kaggle/working/v136') if Path('/kaggle/working').exists() else Path('./v136')
OUT.mkdir(parents=True,exist_ok=True)

@dataclass(frozen=True)
class CFG:
    seed:int=SEED
    fold_seed:int=FOLD_SEED
    n_outer:int=5
    n_inner:int=3
    word_max:int=60000
    char_max:int=80000
    ridge_alpha:float=3.0
    kmeans_k:int=6
    near_threshold:float=0.90
    lambda_grid:tuple=(0.0,0.10,0.20,0.35,0.50,0.75,1.0)
    label_gain:tuple=(0.00,0.25,0.40,0.55,0.70,0.85,1.00)
    rank_estimators:int=300
    rank_lr:float=0.025
    rank_leaves:int=15
    rank_depth:int=4
    rank_min_child:int=80
    rank_subsample:float=0.85
    rank_colsample:float=0.75
    rank_l1:float=0.5
    rank_l2:float=10.0
    rank_max_bin:int=127
    truncation_level:int=8
cfg=CFG()
print(asdict(cfg))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 634.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.0 MB/s eta 0:00:00
{'seed': 20260909, 'fold_seed': 20260907, 'n_outer': 5, 'n_inner': 3, 'word_max': 60000, 'char_max': 80000, 'ridge_alpha': 3.0, 'kmeans_k': 6, 'near_threshold': 0.9, 'lambda_grid': (0.0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0), 'label_gain': (0.0, 0.25, 0.4, 0.55, 0.7, 0.85, 1.0), 'rank_estimators': 300, 'rank_lr': 0.025, 'rank_leaves': 15, 'rank_depth': 4, 'rank_min_child': 80, 'rank_subsample': 0.85, 'rank_colsample': 0.75, 'rank_l1': 0.5, 'rank_l2': 10.0, 'rank_max_bin': 127, 'truncation_level': 8}


In [2]:
# =========================
# 1. Organizer data discovery + strict schema checks
# =========================
def find_file(name):
    candidates=[]
    for root in [Path('/kaggle/input'),Path('/mnt/data'),Path('.')]:
        if root.exists(): candidates.extend(root.rglob(name))
    if not candidates: raise FileNotFoundError(name)
    # Prefer shallow organizer datasets over output copies.
    candidates=sorted(set(candidates), key=lambda p:(len(p.parts),str(p)))
    return candidates[0]

paths={n:find_file(n) for n in ['train_relevance.csv','test.csv','chat_history.csv','user_assessments.csv','modules_catalog.csv','sample_submission.csv']}
print({k:str(v) for k,v in paths.items()})
train=pd.read_csv(paths['train_relevance.csv'])
test=pd.read_csv(paths['test.csv'])
chat=pd.read_csv(paths['chat_history.csv'])
assdf=pd.read_csv(paths['user_assessments.csv'])
modules=pd.read_csv(paths['modules_catalog.csv'])
sample=pd.read_csv(paths['sample_submission.csv'])

MODULES=[c for c in train.columns if c!='user_id']
assert MODULES==[c for c in sample.columns if c!='user_id'] and len(MODULES)==17
assert len(train)==4000 and len(test)==1000 and len(modules)==17 and len(assdf)==5000
assert train.user_id.is_unique and test.user_id.is_unique and not set(train.user_id)&set(test.user_id)
assert list(test.user_id.astype(str))==list(sample.user_id.astype(str))
Y=train[MODULES].to_numpy(np.float32)
assert np.isfinite(Y).all() and (Y>=0).all()
ALL_USERS=np.r_[train.user_id.astype(str).values,test.user_id.astype(str).values]
UID2I={u:i for i,u in enumerate(ALL_USERS)}
NTR,NTE,M=len(train),len(test),len(MODULES)
MOD2I={m:i for i,m in enumerate(MODULES)}

# Exact target grades: rank1=6 ... rank6=1, irrelevant=0.
GRADES=np.zeros_like(Y,dtype=np.int16)
for i,row in enumerate(Y):
    pos=np.flatnonzero(row>0)
    order=pos[np.argsort(-row[pos],kind='stable')]
    for r,j in enumerate(order,1): GRADES[i,j]=7-r
assert set(np.unique(GRADES)).issubset(set(range(7)))
print('K=',dict(zip(*np.unique((Y>0).sum(1),return_counts=True))))
print('grade counts=',dict(zip(*np.unique(GRADES,return_counts=True))))

{'train_relevance.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/train_relevance.csv', 'test.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/test.csv', 'chat_history.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/chat_history.csv', 'user_assessments.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/user_assessments.csv', 'modules_catalog.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/modules_catalog.csv', 'sample_submission.csv': '/kaggle/input/datasets/riantamaputra/mine-today-data-mining-competition-it-today-2026/sample_submission.csv'}
K= {np.int64(4): np.int64(1362), np.int64(5): np.int64(1285), np.int64(6): np.int64(1353)}
grade counts= {np.int16(0): np.int64(48009), np.int16(1): np.int64(1353), np.int16(2): np.int64(2638), np.int16(3): np.int64(4000), np.int

In [3]:
# =========================
# 2. Deterministic text, assessments, generator-state tables
# =========================
import unicodedata

def norm_text(s):
    s='' if pd.isna(s) else str(s)
    s=unicodedata.normalize('NFKC',s).lower()
    s=re.sub(r'https?://\S+',' ',s); s=re.sub(r'[^0-9a-zA-ZÀ-ÿ_]+',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def skeleton(s):
    s=norm_text(s)
    s=re.sub(r'\b\d+(?:\.\d+)?\b','<num>',s)
    return s if s else '<no_chat>'

chat=chat.copy(); chat['timestamp']=pd.to_datetime(chat['timestamp'],errors='coerce'); chat['txt']=chat.user_chat_text.map(norm_text)
chat_valid=chat[chat.user_id.astype(str).isin(set(ALL_USERS))].sort_values(['user_id','timestamp','chat_id'])
by={u:g for u,g in chat_valid.groupby(chat_valid.user_id.astype(str),sort=False)}
FULL_TEXT=[]; LATEST_TEXT=[]; FIRST_TEXT=[]; CHAT_STATE=[]; LATEST_GROUP=[]
for u in ALL_USERS:
    g=by.get(str(u))
    if g is None or len(g)==0:
        full=latest=first='<no_chat>'; n=0; ts=[]
    else:
        texts=g.txt.astype(str).tolist(); full=' [sep] '.join(texts); first=texts[0] or '<empty>'; latest=texts[-1] or '<empty>'; n=len(texts)
        ts=[x for x in g.timestamp.tolist() if not pd.isna(x)]
    FULL_TEXT.append(full); LATEST_TEXT.append(latest); FIRST_TEXT.append(first); LATEST_GROUP.append(skeleton(latest))
    if len(ts)>=1:
        days=np.array([(x-ts[0]).total_seconds()/86400 for x in ts],float); gaps=np.diff(days)
        span=float(days[-1]-days[0]); mean_gap=float(gaps.mean()) if len(gaps) else 0.; max_gap=float(gaps.max()) if len(gaps) else 0.
    else: span=mean_gap=max_gap=0.
    nochat=float(n==0); tier_low=float(1<=n<=2); tier_med=float(3<=n<=5); tier_high=float(n>=6)
    pivot=0.
    if n in (3,4,5) and len(ts)>=3:
        gd=np.diff(np.array([x.timestamp() for x in ts]))/86400
        pivot=float(25<=gd[0]<=60 and np.all(gd[1:]>=3))
    CHAT_STATE.append([nochat,n,tier_low,tier_med,tier_high,span,mean_gap,max_gap,pivot])
FULL_TEXT=np.asarray(FULL_TEXT,object); LATEST_TEXT=np.asarray(LATEST_TEXT,object); FIRST_TEXT=np.asarray(FIRST_TEXT,object); LATEST_GROUP=np.asarray(LATEST_GROUP,object)
CHAT_STATE=np.asarray(CHAT_STATE,np.float32)

# Assessment parser: ten stable questions ordered from first valid JSON row.
ID_NUM={'nol':0,'satu':1,'dua':2,'tiga':3,'empat':4,'lima':5}
parsed={}
questions=[]
for r in assdf.itertuples(index=False):
    try: d=ast.literal_eval(str(r.assessment_result)) if str(r.assessment_result).strip().startswith('{') else json.loads(str(r.assessment_result))
    except Exception: d={}
    parsed[str(r.user_id)]=d
    if not questions and len(d)>=10: questions=list(d.keys())[:10]
assert len(questions)==10
ASS=np.zeros((len(ALL_USERS),10),np.float32); ASS_BAD=np.zeros_like(ASS)
for i,u in enumerate(ALL_USERS):
    d=parsed.get(str(u),{})
    for j,q in enumerate(questions):
        v=d.get(q,np.nan); bad=0.
        if isinstance(v,str):
            vv=v.strip().lower(); bad=1.; v=ID_NUM.get(vv,vv)
        try: x=float(v)
        except Exception: x=np.nan; bad=1.
        if not np.isfinite(x): x=np.nan; bad=1.
        if np.isfinite(x) and not (0<=x<=5): bad=1.
        ASS[i,j]=np.nan if not np.isfinite(x) else np.clip(x,0,5); ASS_BAD[i,j]=bad
ASS_NAN=np.isnan(ASS)

# Catalog documents and simple prerequisite ordinal.
modules=modules.set_index('module_id').loc[MODULES].reset_index()
MOD_TEXT=np.asarray([(norm_text(r.module_name)+' '+norm_text(r.description_and_syllabus)+' '+norm_text(r.prerequisite_level)).strip() for r in modules.itertuples()],object)
def prereq_ord(s):
    s=norm_text(s)
    if 'pemula' in s: return 0
    if 'menengah' in s or 'intermediate' in s: return 1
    if 'lanjut' in s or 'machine learning' in s or 'python' in s: return 2
    return 1
MOD_LEVEL=np.asarray([prereq_ord(x) for x in modules.prerequisite_level],np.float32)
print('text/state ready',len(FULL_TEXT),CHAT_STATE.shape,ASS.shape)

text/state ready 5000 (5000, 9) (5000, 10)


In [4]:
# =========================
# 3. Pinned E5 + Nomic label-free dense similarities (cache-aware, T4x2)
# =========================
E5_ID='intfloat/multilingual-e5-base'; E5_REV='d128750597153bb5987e10b1c3493a34e5a4502a'
NOMIC_ID='nomic-ai/nomic-embed-text-v2-moe'; NOMIC_REV='1066b6599d099fbb93dfcb64f9c37a7c9e503e85'; NOMIC_CODE_REV='7710840340a098cfb869c4f65e87cf2b1b70caca'

def resolve_model(model_id):
    needle=model_id.split('/')[-1].lower().replace('_','-')
    if Path('/kaggle/input').exists():
        for p in Path('/kaggle/input').rglob('config.json'):
            if needle in str(p.parent).lower().replace('_','-'): return str(p.parent)
    return model_id

def load_external_npy(name,rows):
    if not Path('/kaggle/input').exists(): return None
    for p in Path('/kaggle/input').rglob(name):
        try:
            a=np.load(p,mmap_mode='r')
            if a.shape[0]==rows: return np.asarray(a,np.float32)
        except Exception: pass
    return None

def e5_encode(texts,prefix,device,batch):
    from sentence_transformers import SentenceTransformer
    mid=resolve_model(E5_ID)
    model=SentenceTransformer(mid,device=device,revision=None if mid!=E5_ID else E5_REV)
    arr=np.asarray(model.encode([prefix+t for t in texts],batch_size=batch,normalize_embeddings=True,show_progress_bar=True,convert_to_numpy=True),np.float32)
    del model; gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass
    return arr

def nomic_encode(texts,prefix,device,batch):
    import torch
    from transformers import AutoTokenizer,AutoModel
    mid=resolve_model(NOMIC_ID)
    tok=AutoTokenizer.from_pretrained(mid,trust_remote_code=True,revision=None if mid!=NOMIC_ID else NOMIC_REV)
    kw={'trust_remote_code':True,'torch_dtype':torch.float16 if torch.cuda.is_available() else torch.float32}
    if mid==NOMIC_ID: kw.update(revision=NOMIC_REV,code_revision=NOMIC_CODE_REV)
    model=AutoModel.from_pretrained(mid,**kw).to(device).eval(); out=[]
    with torch.no_grad():
        for a in range(0,len(texts),batch):
            e=tok([prefix+t for t in texts[a:a+batch]],padding=True,truncation=True,max_length=256,return_tensors='pt'); e={k:v.to(device) for k,v in e.items()}
            h=model(**e).last_hidden_state; mask=e['attention_mask'].unsqueeze(-1).float(); v=(h*mask).sum(1)/mask.sum(1).clamp_min(1)
            v=v.float(); v=v/v.norm(dim=1,keepdim=True).clamp_min(1e-12); out.append(v.cpu().numpy())
    del model; gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(out).astype(np.float32)

try:
    import torch
    d0='cuda:0' if torch.cuda.is_available() else 'cpu'; d1='cuda:1' if torch.cuda.is_available() and torch.cuda.device_count()>1 else d0
except Exception: d0=d1='cpu'
E5U=load_external_npy('users_e5.npy',len(ALL_USERS)); E5M=load_external_npy('modules_e5.npy',M)
if E5U is None or E5M is None:
    E5U=e5_encode(FULL_TEXT.tolist(),'query: ',d0,64); E5M=e5_encode(MOD_TEXT.tolist(),'passage: ',d0,17)
    np.save(OUT/'users_e5.npy',E5U); np.save(OUT/'modules_e5.npy',E5M)
NOU=load_external_npy('users_nomic.npy',len(ALL_USERS)); NOM=load_external_npy('modules_nomic.npy',M)
if NOU is None or NOM is None:
    NOU=nomic_encode(FULL_TEXT.tolist(),'search_query: ',d1,48); NOM=nomic_encode(MOD_TEXT.tolist(),'search_document: ',d1,17)
    np.save(OUT/'users_nomic.npy',NOU); np.save(OUT/'modules_nomic.npy',NOM)
for a in [E5U,E5M,NOU,NOM]:
    a/=np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-12)
E5_SIM=(E5U@E5M.T).astype(np.float32); NOM_SIM=(NOU@NOM.T).astype(np.float32)
json.dump({'e5':{'id':E5_ID,'revision':E5_REV},'nomic':{'id':NOMIC_ID,'revision':NOMIC_REV,'code_revision':NOMIC_CODE_REV},'shapes':[list(x.shape) for x in [E5U,E5M,NOU,NOM]]},open(OUT/'v136_dense_meta.json','w'),indent=2)
print('dense similarities',E5_SIM.shape,NOM_SIM.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

configuration_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_hf_nomic_bert.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.90G [00:00<?, ?B/s]

dense similarities (5000, 17) (5000, 17)


In [5]:
# =========================
# 4. Metrics, fold registry, lexical/BM25 utilities, Stage-1 cross-fitting
# =========================
def ndcg_rows(y_true,scores,k=5):
    y_true=np.asarray(y_true,float); scores=np.asarray(scores,float)
    ordp=np.argsort(-scores,axis=1,kind='stable')[:,:k]
    disc=1/np.log2(np.arange(2,k+2))
    dcg=np.sum(np.take_along_axis(y_true,ordp,axis=1)*disc,axis=1)
    ideal=np.sort(y_true,axis=1)[:,::-1][:,:k]; idcg=np.sum(ideal*disc,axis=1)
    return np.divide(dcg,idcg,out=np.zeros_like(dcg),where=idcg>0)

def strata(rows):
    yy=Y[np.asarray(rows,int)]; top=np.argmax(yy,1); kk=(yy>0).sum(1)
    raw=np.asarray([f'{a}_{b}' for a,b in zip(top,kk)],object); c=Counter(raw)
    return np.asarray([x if c[x]>=5 else str(a) for x,a in zip(raw,top)],object)

def world_splits(world,seed=FOLD_SEED):
    rows=np.arange(NTR); st=strata(rows)
    if world=='random':
        sk=StratifiedKFold(5,shuffle=True,random_state=seed); return [(rows[a],rows[b]) for a,b in sk.split(rows,st)]
    if world=='template':
        sg=StratifiedGroupKFold(5,shuffle=True,random_state=seed); return [(rows[a],rows[b]) for a,b in sg.split(rows,st,LATEST_GROUP[:NTR])]
    if world=='contiguous':
        blocks=np.array_split(rows,5); return [(np.setdiff1d(rows,b),b) for b in blocks]
    raise ValueError(world)

def inner_splits(rows,seed):
    rows=np.asarray(rows,int); groups=LATEST_GROUP[rows]; st=strata(rows)
    try:
        sg=StratifiedGroupKFold(3,shuffle=True,random_state=seed)
        return [(rows[a],rows[b]) for a,b in sg.split(rows,st,groups)]
    except Exception:
        g=GroupKFold(3); return [(rows[a],rows[b]) for a,b in g.split(rows,groups=groups)]

def clean_ass_fit(ref,query):
    med=np.nanmedian(ASS[ref],axis=0); med=np.where(np.isfinite(med),med,0)
    tr=np.where(np.isnan(ASS[ref]),med,ASS[ref]); q=np.where(np.isnan(ASS[query]),med,ASS[query])
    sc=StandardScaler().fit(tr); return sc.transform(q).astype(np.float32),med,sc

def stage1_ridge_fit(train_rows,query_rows):
    tr=np.asarray(train_rows,int); qr=np.asarray(query_rows,int)
    # Fold-fit word+char text and assessments; multioutput exact relevance and Top1 heads.
    w=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=cfg.word_max,sublinear_tf=True,dtype=np.float32)
    c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=cfg.char_max,sublinear_tf=True,dtype=np.float32)
    Xw=w.fit_transform(FULL_TEXT[tr]); Xc=c.fit_transform(FULL_TEXT[tr]); Qw=w.transform(FULL_TEXT[qr]); Qc=c.transform(FULL_TEXT[qr])
    Atr,med,sc=clean_ass_fit(tr,tr); Aq=np.where(np.isnan(ASS[qr]),med,ASS[qr]); Aq=sc.transform(Aq).astype(np.float32)
    X=sparse.hstack([Xw,Xc,sparse.csr_matrix(Atr),sparse.csr_matrix(ASS_BAD[tr])],format='csr'); Q=sparse.hstack([Qw,Qc,sparse.csr_matrix(Aq),sparse.csr_matrix(ASS_BAD[qr])],format='csr')
    regr=Ridge(alpha=cfg.ridge_alpha,solver='lsqr').fit(X,Y[tr]); exact=regr.predict(Q).astype(np.float32)
    top=(np.arange(M)[None,:]==np.argmax(Y[tr],axis=1)[:,None]).astype(np.float32)
    rt=Ridge(alpha=cfg.ridge_alpha,solver='lsqr').fit(X,top); topp=rt.predict(Q).astype(np.float32)
    return exact,topp

def stage1_oof(rows,seed):
    rows=np.asarray(rows,int); pos={g:i for i,g in enumerate(rows)}; A=np.zeros((len(rows),M),np.float32); T=np.zeros_like(A)
    for a,b in inner_splits(rows,seed):
        p,q=stage1_ridge_fit(a,b)
        for j,g in enumerate(b): A[pos[g]]=p[j]; T[pos[g]]=q[j]
    assert np.isfinite(A).all(); return A,T

def tokenize(s): return re.findall(r'[a-z0-9_]+',norm_text(s))
def bm25_matrix(user_texts):
    # Fixed module corpus; no target information. k1=1.2,b=.75, unigram.
    docs=[tokenize(x) for x in MOD_TEXT]; N=len(docs); lens=np.array([len(x) for x in docs],float); av=lens.mean(); df=Counter()
    for d in docs:
        for t in set(d): df[t]+=1
    idf={t:math.log(1+(N-n+0.5)/(n+0.5)) for t,n in df.items()}; out=np.zeros((len(user_texts),M),np.float32)
    for i,qtxt in enumerate(user_texts):
        q=set(tokenize(qtxt))
        for j,d in enumerate(docs):
            tf=Counter(d); z=0.
            for t in q:
                f=tf.get(t,0)
                if f: z+=idf.get(t,0)*f*2.2/(f+1.2*(1-.75+.75*lens[j]/av))
            out[i,j]=z
    return out
BM25_FULL=bm25_matrix(FULL_TEXT); BM25_LATEST=bm25_matrix(LATEST_TEXT)
print('fold/metric helpers ready')

fold/metric helpers ready


In [6]:
# =========================
# 5. Fold-local candidate feature builder (C1 vs C2)
# =========================
def lexical_sim_fit(ref,query):
    ref=np.asarray(ref,int); query=np.asarray(query,int)
    # Word and char spaces are fold-fit on outer-train users plus the 17 module documents.
    w=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=cfg.word_max,sublinear_tf=True,norm='l2',dtype=np.float32)
    c=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=cfg.char_max,sublinear_tf=True,norm='l2',dtype=np.float32)
    w.fit(list(FULL_TEXT[ref])+list(MOD_TEXT)); c.fit(list(FULL_TEXT[ref])+list(MOD_TEXT))
    MW=w.transform(MOD_TEXT); MC=c.transform(MOD_TEXT)
    vals=[]
    for texts in [FULL_TEXT[query],LATEST_TEXT[query]]:
        W=w.transform(texts); C=c.transform(texts); vals.extend([(W@MW.T).toarray().astype(np.float32),(C@MC.T).toarray().astype(np.float32)])
    return vals  # full_word, full_char, latest_word, latest_char

def fit_unsup_state(ref,query):
    ref=np.asarray(ref,int); query=np.asarray(query,int)
    med=np.nanmedian(ASS[ref],0); med=np.where(np.isfinite(med),med,0)
    Atr=np.where(np.isnan(ASS[ref]),med,ASS[ref]); Aq=np.where(np.isnan(ASS[query]),med,ASS[query])
    sc=StandardScaler().fit(Atr); Ztr=sc.transform(Atr); Zq=sc.transform(Aq)
    km=KMeans(n_clusters=cfg.kmeans_k,n_init=50,random_state=cfg.seed).fit(Ztr)
    dist=km.transform(Zq).astype(np.float32); nearest=np.eye(cfg.kmeans_k,dtype=np.float32)[np.argmin(dist,1)]
    # template frequency is outer-train-only
    cnt=Counter(LATEST_GROUP[ref]); same_ref_query=np.array_equal(ref,query)
    tf=np.asarray([math.log1p(max(0,cnt.get(g,0)-(1 if same_ref_query else 0))) for g in LATEST_GROUP[query]],np.float32)[:,None]
    # exact/near-duplicate indicators against fold-train text; leave self out on training rows.
    vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=1,max_features=50000,norm='l2',dtype=np.float32)
    X=vec.fit_transform([skeleton(x) for x in FULL_TEXT[ref]]); Q=vec.transform([skeleton(x) for x in FULL_TEXT[query]])
    nn=NearestNeighbors(n_neighbors=2 if same_ref_query and len(ref)>1 else 1,metric='cosine',algorithm='brute').fit(X); d,_=nn.kneighbors(Q); mx=(1-d[:,1] if same_ref_query and d.shape[1]>1 else 1-d[:,0]).astype(np.float32)
    near=np.c_[mx,(mx>=cfg.near_threshold).astype(np.float32)]
    cleanq=np.where(np.isnan(ASS[query]),med,ASS[query]).astype(np.float32)
    readiness=np.c_[cleanq.mean(1),cleanq.std(1),cleanq.max(1),cleanq.min(1),ASS_BAD[query].sum(1)]
    return np.hstack([cleanq,ASS_BAD[query],readiness,dist,nearest,CHAT_STATE[query],tf,near]).astype(np.float32)

def user_module_rows(U, module_features):
    # U: [n,d] -> repeated candidate rows; module_features: [M,q]
    n=len(U); return np.hstack([np.repeat(U,M,axis=0),np.tile(module_features,(n,1))]).astype(np.float32)

def build_features(ref,query,A_query,T_query,include_state):
    ref=np.asarray(ref,int); query=np.asarray(query,int); n=len(query)
    fw,fc,lw,lc=lexical_sim_fit(ref,query)
    # candidate row level dynamic features
    dense=np.stack([E5_SIM[query],NOM_SIM[query],fw,fc,lw,lc,BM25_FULL[query],BM25_LATEST[query],A_query,T_query],axis=2)
    # A geometry within user
    order=np.argsort(-A_query,axis=1,kind='stable'); rank=np.empty_like(order)
    for i in range(n): rank[i,order[i]]=np.arange(M)
    sortA=np.take_along_axis(A_query,order,axis=1); top=sortA[:,0]; fifth=sortA[:,4]
    geom=np.zeros((n,M,5),np.float32)
    for i in range(n):
        for j in range(M):
            r=rank[i,j]; geom[i,j]=[r/16.,A_query[i,j]-top[i],A_query[i,j]-fifth[i], top[i]-sortA[i,1], (A_query[i,j]-A_query[i,order[i,min(r+1,M-1)]])]
    base=np.concatenate([dense,geom],axis=2).reshape(n*M,-1)
    onehot=np.eye(M,dtype=np.float32); modfeat=np.c_[onehot,MOD_LEVEL[:,None]]
    X=np.hstack([base,np.tile(modfeat,(n,1))])
    # Fold-train module prevalences only.
    prev=Y[ref].mean(0).astype(np.float32); X=np.hstack([X,np.tile(prev[:,None],(n,1))])
    if include_state:
        U=fit_unsup_state(ref,query); X=np.hstack([X,user_module_rows(U,np.zeros((M,0),np.float32))])
    assert X.shape[0]==n*M and np.isfinite(X).all()
    return X.astype(np.float32)

In [7]:
# =========================
# 6. Model definitions, within-user fusion, primary outer CV
# =========================
def rank_params(seed):
    return dict(objective='lambdarank',n_estimators=cfg.rank_estimators,learning_rate=cfg.rank_lr,num_leaves=cfg.rank_leaves,max_depth=cfg.rank_depth,
                min_child_samples=cfg.rank_min_child,subsample=cfg.rank_subsample,subsample_freq=1,colsample_bytree=cfg.rank_colsample,
                reg_alpha=cfg.rank_l1,reg_lambda=cfg.rank_l2,max_bin=cfg.rank_max_bin,deterministic=True,force_col_wise=True,
                random_state=int(seed),n_jobs=4,verbosity=-1,label_gain=list(cfg.label_gain),lambdarank_truncation_level=cfg.truncation_level)
def reg_params(seed):
    p=rank_params(seed); p.pop('objective'); p.pop('label_gain'); p.pop('lambdarank_truncation_level'); p.update(objective='regression_l2')
    return p

def fit_c0(X,yflat,seed):
    return lgb.LGBMRegressor(**reg_params(seed)).fit(X,yflat)
def fit_rank(X,gflat,nusers,seed):
    return lgb.LGBMRanker(**rank_params(seed)).fit(X,gflat,group=[M]*nusers)
def zrow(S):
    S=np.asarray(S,np.float32); mu=S.mean(1,keepdims=True); sd=S.std(1,keepdims=True); return (S-mu)/np.maximum(sd,1e-6)
def fused(A,R,lam): return (1-lam)*zrow(A)+lam*zrow(R)
def choose_lambda(rows,A,R):
    vals=[]
    for lam in cfg.lambda_grid:
        vals.append(float(ndcg_rows(Y[rows],fused(A,R,lam)).mean()))
    best=max(vals); idx=min(i for i,v in enumerate(vals) if abs(v-best)<1e-12)
    return float(cfg.lambda_grid[idx]),vals

def select_lambda_nested(rows,seed):
    rows=np.asarray(rows,int); pos={g:i for i,g in enumerate(rows)}
    AA=np.zeros((len(rows),M),np.float32); R1=np.zeros_like(AA); R2=np.zeros_like(AA)
    for q,(it,iv) in enumerate(inner_splits(rows,seed)):
        A_it,T_it=stage1_oof(it,seed+100+q); A_iv,T_iv=stage1_ridge_fit(it,iv)
        X1it=build_features(it,it,A_it,T_it,False); X1iv=build_features(it,iv,A_iv,T_iv,False)
        X2it=build_features(it,it,A_it,T_it,True); X2iv=build_features(it,iv,A_iv,T_iv,True)
        m1=fit_rank(X1it,GRADES[it].reshape(-1),len(it),seed+200+q)
        m2=fit_rank(X2it,GRADES[it].reshape(-1),len(it),seed+300+q)
        P1=m1.predict(X1iv).reshape(len(iv),M); P2=m2.predict(X2iv).reshape(len(iv),M)
        for j,g in enumerate(iv):
            k=pos[g]; AA[k]=A_iv[j]; R1[k]=P1[j]; R2[k]=P2[j]
        del X1it,X1iv,X2it,X2iv,m1,m2; gc.collect()
    l1,v1=choose_lambda(rows,AA,R1); l2,v2=choose_lambda(rows,AA,R2)
    return l1,l2,v1,v2

def primary_cv():
    folds=world_splits('random'); rec=[]; oof={k:np.zeros((NTR,M),np.float32) for k in ['c0','c1','c2']}; fold_id=np.full(NTR,-1,int); lambdas=[]
    for f,(tr,va) in enumerate(folds):
        print(f'PRIMARY fold {f}: {len(tr)}/{len(va)}'); t0=time.time()
        l1,l2,v1,v2=select_lambda_nested(tr,cfg.seed+4000+f); lambdas.append((l1,l2))
        A_tr,T_tr=stage1_oof(tr,cfg.seed+1000*f); A_va,T_va=stage1_ridge_fit(tr,va)
        X1tr=build_features(tr,tr,A_tr,T_tr,False); X1va=build_features(tr,va,A_va,T_va,False)
        X2tr=build_features(tr,tr,A_tr,T_tr,True); X2va=build_features(tr,va,A_va,T_va,True)
        yflat=Y[tr].reshape(-1); gflat=GRADES[tr].reshape(-1)
        c0=fit_c0(X1tr,yflat,cfg.seed+10*f); c1=fit_rank(X1tr,gflat,len(tr),cfg.seed+20*f); c2=fit_rank(X2tr,gflat,len(tr),cfg.seed+30*f)
        P0=c0.predict(X1va).reshape(len(va),M).astype(np.float32); R1=c1.predict(X1va).reshape(len(va),M).astype(np.float32); R2=c2.predict(X2va).reshape(len(va),M).astype(np.float32)
        S1=fused(A_va,R1,l1); S2=fused(A_va,R2,l2)
        oof['c0'][va]=P0; oof['c1'][va]=S1; oof['c2'][va]=S2; fold_id[va]=f
        m0=ndcg_rows(Y[va],P0).mean(); m1v=ndcg_rows(Y[va],S1).mean(); m2v=ndcg_rows(Y[va],S2).mean()
        rec.append({'fold':f,'c0':m0,'c1':m1v,'c2':m2v,'c1_c0':m1v-m0,'c2_c1':m2v-m1v,'c2_c0':m2v-m0,'lambda_c1':l1,'lambda_c2':l2,'inner_c1_best':max(v1),'inner_c2_best':max(v2),'sec':time.time()-t0})
        print(rec[-1]); del X1tr,X1va,X2tr,X2va,c0,c1,c2; gc.collect()
    return pd.DataFrame(rec),oof,fold_id,lambdas
PRIMARY,OOF,FOLD_ID,LAMBDAS=primary_cv()
PRIMARY.to_csv(OUT/'v136_primary_fold_metrics.csv',index=False); np.savez_compressed(OUT/'v136_oof_scores.npz',**OOF,fold_id=FOLD_ID)
print(PRIMARY); print('means',PRIMARY[['c0','c1','c2','c1_c0','c2_c1','c2_c0']].mean())


PRIMARY fold 0: 3200/800
{'fold': 0, 'c0': np.float64(0.6553801934731854), 'c1': np.float64(0.654376824026251), 'c2': np.float64(0.6569733718011606), 'c1_c0': np.float64(-0.0010033694469344656), 'c2_c1': np.float64(0.0025965477749095855), 'c2_c0': np.float64(0.00159317832797512), 'lambda_c1': 0.75, 'lambda_c2': 1.0, 'inner_c1_best': 0.6511802436755579, 'inner_c2_best': 0.6521553359325776, 'sec': 132.1973373889923}
PRIMARY fold 1: 3200/800
{'fold': 1, 'c0': np.float64(0.6532905489179541), 'c1': np.float64(0.6541569231218396), 'c2': np.float64(0.6545700142151896), 'c1_c0': np.float64(0.0008663742038854672), 'c2_c1': np.float64(0.00041309109334997096), 'c2_c0': np.float64(0.0012794652972354381), 'lambda_c1': 0.75, 'lambda_c2': 0.75, 'inner_c1_best': 0.6484655720707031, 'inner_c2_best': 0.6495473637546141, 'sec': 131.75392365455627}
PRIMARY fold 2: 3200/800
{'fold': 2, 'c0': np.float64(0.6636255791527645), 'c1': np.float64(0.6617716683181544), 'c2': np.float64(0.6631982298806578), 'c1_c0':

In [8]:
# =========================
# 7. Fixed-lambda stress worlds: template, contiguous, explicit forward holdout
# =========================
def modal_lambda(col):
    vals=[x[0 if col=='c1' else 1] for x in LAMBDAS]; c=Counter(vals); mx=max(c.values()); return min(v for v,n in c.items() if n==mx)
LAM1,LAM2=modal_lambda('c1'),modal_lambda('c2')
print('frozen lambdas',LAM1,LAM2)

def eval_split(tr,va,seed,tag):
    A_tr,T_tr=stage1_oof(tr,seed+1); A_va,T_va=stage1_ridge_fit(tr,va)
    X1tr=build_features(tr,tr,A_tr,T_tr,False); X1va=build_features(tr,va,A_va,T_va,False); X2tr=build_features(tr,tr,A_tr,T_tr,True); X2va=build_features(tr,va,A_va,T_va,True)
    c0=fit_c0(X1tr,Y[tr].reshape(-1),seed+2); c1=fit_rank(X1tr,GRADES[tr].reshape(-1),len(tr),seed+3); c2=fit_rank(X2tr,GRADES[tr].reshape(-1),len(tr),seed+4)
    P0=c0.predict(X1va).reshape(len(va),M); P1=fused(A_va,c1.predict(X1va).reshape(len(va),M),LAM1); P2=fused(A_va,c2.predict(X2va).reshape(len(va),M),LAM2)
    a=ndcg_rows(Y[va],P0); b=ndcg_rows(Y[va],P1); c=ndcg_rows(Y[va],P2)
    out={'tag':tag,'n':len(va),'c0':a.mean(),'c1':b.mean(),'c2':c.mean(),'c1_c0':(b-a).mean(),'c2_c1':(c-b).mean(),'c2_c0':(c-a).mean()}
    del X1tr,X1va,X2tr,X2va,c0,c1,c2; gc.collect(); return out

stress=[]
for world in ['template','contiguous']:
    for f,(tr,va) in enumerate(world_splits(world)):
        print(world,f); stress.append(eval_split(tr,va,cfg.seed+10000+f+(0 if world=='template' else 1000),f'{world}_{f}'))
# Forward 1..3200 -> 3201..4000, exact contract.
stress.append(eval_split(np.arange(3200),np.arange(3200,4000),cfg.seed+13000,'forward_3201_4000'))
STRESS=pd.DataFrame(stress); STRESS.to_csv(OUT/'v136_stress_metrics.csv',index=False); print(STRESS)

frozen lambdas 0.75 1.0
template 0
template 1
template 2
template 3
template 4
contiguous 0
contiguous 1
contiguous 2
contiguous 3
contiguous 4
                  tag    n        c0        c1        c2     c1_c0     c2_c1  \
0          template_0  836  0.623090  0.622729  0.622094 -0.000361 -0.000635   
1          template_1  796  0.663741  0.663358  0.666257 -0.000384  0.002899   
2          template_2  807  0.652862  0.655113  0.655404  0.002251  0.000291   
3          template_3  803  0.662887  0.662836  0.664762 -0.000051  0.001926   
4          template_4  758  0.659691  0.659599  0.660961 -0.000092  0.001362   
5        contiguous_0  800  0.653041  0.656766  0.655336  0.003725 -0.001430   
6        contiguous_1  800  0.648957  0.649406  0.648888  0.000450 -0.000519   
7        contiguous_2  800  0.668397  0.667834  0.668009 -0.000563  0.000174   
8        contiguous_3  800  0.652325  0.654225  0.655636  0.001900  0.001412   
9        contiguous_4  800  0.653470  0.652554  0.654407

In [9]:
# =========================
# 8. Uncertainty, behavior slices, objective/generator ablation report
# =========================
def cluster_bootstrap(delta,groups,B=2000,seed=SEED):
    rng=np.random.default_rng(seed); groups=np.asarray(groups,object); uniq=np.unique(groups); mp={g:np.flatnonzero(groups==g) for g in uniq}; vals=[]
    for _ in range(B):
        gs=rng.choice(uniq,size=len(uniq),replace=True); idx=np.concatenate([mp[g] for g in gs]); vals.append(float(np.mean(delta[idx])))
    return [float(np.quantile(vals,.025)),float(np.quantile(vals,.975))]

d0=ndcg_rows(Y,OOF['c0']); d1=ndcg_rows(Y,OOF['c1']); d2=ndcg_rows(Y,OOF['c2'])
d10=d1-d0; d21=d2-d1; d20=d2-d0
summary={
 'c0_mean':float(d0.mean()),'c1_mean':float(d1.mean()),'c2_mean':float(d2.mean()),
 'c1_c0':float(d10.mean()),'c2_c1':float(d21.mean()),'c2_c0':float(d20.mean()),
 'c1_c0_positive_folds':int((PRIMARY.c1_c0>0).sum()),'c2_c1_nonnegative_folds':int((PRIMARY.c2_c1>=0).sum()),'c2_c0_positive_folds':int((PRIMARY.c2_c0>0).sum()),
 'c1_c0_cluster_ci':cluster_bootstrap(d10,LATEST_GROUP[:NTR]),'c2_c0_cluster_ci':cluster_bootstrap(d20,LATEST_GROUP[:NTR],seed=SEED+1),
 'lambdas_c1':[float(x[0]) for x in LAMBDAS],'lambdas_c2':[float(x[1]) for x in LAMBDAS],'modal_lambda_c1':LAM1,'modal_lambda_c2':LAM2
}
# Regime slices (diagnostic only)
reg=np.where(CHAT_STATE[:NTR,0]>0,'NO_CHAT',np.where(CHAT_STATE[:NTR,4]>0,'HIGH',np.where(CHAT_STATE[:NTR,3]>0,'MEDIUM','LOW')))
rows=[]
for name in sorted(set(reg)):
    ii=np.flatnonzero(reg==name); rows.append({'slice':name,'n':len(ii),'c0':d0[ii].mean(),'c1':d1[ii].mean(),'c2':d2[ii].mean(),'c2_c0':d20[ii].mean()})
SLICE=pd.DataFrame(rows); SLICE.to_csv(OUT/'v136_slice_metrics.csv',index=False)
json.dump(summary,open(OUT/'v136_validation_summary.json','w'),indent=2)
print(json.dumps(summary,indent=2)); print(SLICE)

{
  "c0_mean": 0.6565684497632989,
  "c1_mean": 0.656858377895717,
  "c2_mean": 0.6583500384082627,
  "c1_c0": 0.00028992813241804274,
  "c2_c1": 0.0014916605125457202,
  "c2_c0": 0.001781588644963763,
  "c1_c0_positive_folds": 3,
  "c2_c1_nonnegative_folds": 5,
  "c2_c0_positive_folds": 4,
  "c1_c0_cluster_ci": [
    -0.0010412499061742391,
    0.0015852787044703803
  ],
  "c2_c0_cluster_ci": [
    0.00020337590885592427,
    0.003289115942510827
  ],
  "lambdas_c1": [
    0.75,
    0.75,
    0.75,
    0.75,
    0.5
  ],
  "lambdas_c2": [
    1.0,
    0.75,
    1.0,
    1.0,
    1.0
  ],
  "modal_lambda_c1": 0.75,
  "modal_lambda_c2": 1.0
}
     slice     n        c0        c1        c2     c2_c0
0     HIGH   422  0.683777  0.685022  0.685473  0.001697
1      LOW  2298  0.647843  0.648471  0.649953  0.002110
2   MEDIUM  1163  0.677601  0.676310  0.677860  0.000258
3  NO_CHAT   117  0.520739  0.526652  0.531518  0.010778


In [10]:
# =========================
# 9. Frozen gate evaluation + final full-data fit/test inference only when research gates permit
# =========================
# Gates are intentionally conservative for a whole-list model.
random_gain=float(PRIMARY.c2_c0.mean())
ordinal_gain=float(PRIMARY.c1_c0.mean())
gen_gain=float(PRIMARY.c2_c1.mean())
template_mean=float(STRESS[STRESS.tag.str.startswith('template_')].c2_c0.mean())
contig_mean=float(STRESS[STRESS.tag.str.startswith('contiguous_')].c2_c0.mean())
forward=float(STRESS.loc[STRESS.tag=='forward_3201_4000','c2_c0'].iloc[0])
worst=float(PRIMARY.c2_c0.min())
# C1 is allowed to become final if C2 adds no positive generator increment.
USE_C2=gen_gain>0
final_col='c2' if USE_C2 else 'c1'; final_delta=(d20 if USE_C2 else d10); final_fold=(PRIMARY.c2_c0 if USE_C2 else PRIMARY.c1_c0)
final_random=float(final_delta.mean())
final_positive_folds=int((final_fold>0).sum())
final_ci=cluster_bootstrap(final_delta,LATEST_GROUP[:NTR],seed=SEED+10)

GATES={
 'correctness':bool(np.isfinite(PRIMARY[['c0','c1','c2']].to_numpy()).all()),
 'ordinal_mechanism_c1_c0_ge_00030':ordinal_gain>=0.00030,
 'ordinal_positive_4of5':int((PRIMARY.c1_c0>0).sum())>=4,
 'generator_nonharm_or_fallback_to_c1':True,
 'final_vs_clean_anchor_ge_00125':final_random>=0.00125,
 'final_positive_4of5':final_positive_folds>=4,
 'worst_fold_ge_minus_00075':float(final_fold.min())>=-0.00075,
 'cluster_ci_lower_positive':final_ci[0]>0,
 'contiguous_mean_ge_00050':(contig_mean if USE_C2 else float(STRESS[STRESS.tag.str.startswith('contiguous_')].c1_c0.mean()))>=0.00050,
 'forward_ge_minus_00100':(forward if USE_C2 else float(STRESS.loc[STRESS.tag=='forward_3201_4000','c1_c0'].iloc[0]))>=-0.00100,
 'lambda_nonzero_4of5':sum((x[1] if USE_C2 else x[0])>0 for x in LAMBDAS)>=4,
}
PRESEED_PASS=all(GATES.values()); SECOND_SEED=[]
if PRESEED_PASS:
    print('All primary gates passed; running required second fixed seed sign check...')
    for f,(tr,va) in enumerate(world_splits('random')):
        z=eval_split(tr,va,20260910+200*f,f'seed2_{f}')
        SECOND_SEED.append(z)
    col='c2_c0' if USE_C2 else 'c1_c0'
    GATES['second_seed_positive_4of5']=sum(float(z[col])>0 for z in SECOND_SEED)>=4
else:
    GATES['second_seed_positive_4of5']=False
GATES['all_submission_gates']=all(GATES.values())
status='SUBMIT_CANDIDATE' if GATES['all_submission_gates'] else 'HOLD'
report={'status':status,'selected_arm':final_col,'public_benchmark_v132':0.66219,'metrics':summary,'stress':STRESS.to_dict('records'),'second_seed':SECOND_SEED,'gates':GATES}
json.dump(report,open(OUT/'v136_gate_report.json','w'),indent=2)
print(json.dumps(report,indent=2))

if status=='SUBMIT_CANDIDATE':
    ALLTR=np.arange(NTR); ALLTE=np.arange(NTR,NTR+NTE)
    A_tr,T_tr=stage1_oof(ALLTR,cfg.seed+20000); A_te,T_te=stage1_ridge_fit(ALLTR,ALLTE)
    X1tr=build_features(ALLTR,ALLTR,A_tr,T_tr,False); X1te=build_features(ALLTR,ALLTE,A_te,T_te,False)
    c0=fit_c0(X1tr,Y.reshape(-1),cfg.seed+20500); BASE=c0.predict(X1te).reshape(NTE,M)
    if USE_C2:
        Xftr=build_features(ALLTR,ALLTR,A_tr,T_tr,True); Xfte=build_features(ALLTR,ALLTE,A_te,T_te,True)
        model=fit_rank(Xftr,GRADES.reshape(-1),NTR,cfg.seed+21000); R=model.predict(Xfte).reshape(NTE,M); S=fused(A_te,R,LAM2)
    else:
        Xftr,Xfte=X1tr,X1te
        model=fit_rank(X1tr,GRADES.reshape(-1),NTR,cfg.seed+21000); R=model.predict(X1te).reshape(NTE,M); S=fused(A_te,R,LAM1)
    order=np.argsort(-S,axis=1,kind='stable'); final_scores=np.zeros_like(S,np.float32); vals=np.linspace(1.0,0.0,M,dtype=np.float32)
    for i in range(NTE): final_scores[i,order[i]]=vals
    sub=sample.copy(); sub[MODULES]=final_scores; sub.to_csv(OUT/'candidate_v136_GC_OWLF.csv',index=False)
    bo=np.argsort(-BASE,axis=1,kind='stable'); fo=np.argsort(-S,axis=1,kind='stable')
    top1chg=(bo[:,0]!=fo[:,0]); top5chg=np.array([set(bo[i,:5])!=set(fo[i,:5]) for i in range(NTE)])
    mu_tr=Xftr.mean(0); sd_tr=Xftr.std(0); smd=(Xfte.mean(0)-mu_tr)/np.maximum(sd_tr,1e-6)
    geom={'top1_changes':int(top1chg.sum()),'top5_membership_changes':int(top5chg.sum()),'max_abs_feature_smd':float(np.max(np.abs(smd))),'mean_abs_feature_smd':float(np.mean(np.abs(smd)))}
    pd.DataFrame({'user_id':test.user_id,'base_top1':[MODULES[j] for j in bo[:,0]],'v136_top1':[MODULES[j] for j in fo[:,0]],'top1_changed':top1chg,'top5_changed':top5chg,'margin':np.sort(S,axis=1)[:,-1]-np.sort(S,axis=1)[:,-2]}).to_csv(OUT/'v136_test_geometry.csv',index=False)
    json.dump(geom,open(OUT/'v136_test_shift.json','w'),indent=2)
    sha=hashlib.sha256((OUT/'candidate_v136_GC_OWLF.csv').read_bytes()).hexdigest(); print('candidate sha256',sha,geom)
else:
    print('V136 HOLD: no new submission candidate produced. Protected public benchmark remains V132=0.66219.')


{
  "status": "HOLD",
  "selected_arm": "c2",
  "public_benchmark_v132": 0.66219,
  "metrics": {
    "c0_mean": 0.6565684497632989,
    "c1_mean": 0.656858377895717,
    "c2_mean": 0.6583500384082627,
    "c1_c0": 0.00028992813241804274,
    "c2_c1": 0.0014916605125457202,
    "c2_c0": 0.001781588644963763,
    "c1_c0_positive_folds": 3,
    "c2_c1_nonnegative_folds": 5,
    "c2_c0_positive_folds": 4,
    "c1_c0_cluster_ci": [
      -0.0010412499061742391,
      0.0015852787044703803
    ],
    "c2_c0_cluster_ci": [
      0.00020337590885592427,
      0.003289115942510827
    ],
    "lambdas_c1": [
      0.75,
      0.75,
      0.75,
      0.75,
      0.5
    ],
    "lambdas_c2": [
      1.0,
      0.75,
      1.0,
      1.0,
      1.0
    ],
    "modal_lambda_c1": 0.75,
    "modal_lambda_c2": 1.0
  },
  "stress": [
    {
      "tag": "template_0",
      "n": 836,
      "c0": 0.6230899801479471,
      "c1": 0.6227287522099482,
      "c2": 0.6220938006776641,
      "c1_c0": -0.000361227

In [11]:
# =========================
# 10. Reproducibility / artifact manifest
# =========================
manifest={'seed':SEED,'fold_seed':FOLD_SEED,'config':asdict(cfg),'python':platform.python_version(),'platform':platform.platform(),'files':{}}
for p in sorted(OUT.iterdir()):
    if p.is_file(): manifest['files'][p.name]={'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()}
try:
    import importlib.metadata as im
    manifest['packages']={x:im.version(x) for x in ['numpy','pandas','scikit-learn','scipy','lightgbm','torch','transformers','sentence-transformers']}
except Exception: pass
json.dump(manifest,open(OUT/'v136_manifest.json','w'),indent=2)
print(json.dumps(manifest,indent=2))
print('OUTPUT DIR:',OUT)

{
  "seed": 20260909,
  "fold_seed": 20260907,
  "config": {
    "seed": 20260909,
    "fold_seed": 20260907,
    "n_outer": 5,
    "n_inner": 3,
    "word_max": 60000,
    "char_max": 80000,
    "ridge_alpha": 3.0,
    "kmeans_k": 6,
    "near_threshold": 0.9,
    "lambda_grid": [
      0.0,
      0.1,
      0.2,
      0.35,
      0.5,
      0.75,
      1.0
    ],
    "label_gain": [
      0.0,
      0.25,
      0.4,
      0.55,
      0.7,
      0.85,
      1.0
    ],
    "rank_estimators": 300,
    "rank_lr": 0.025,
    "rank_leaves": 15,
    "rank_depth": 4,
    "rank_min_child": 80,
    "rank_subsample": 0.85,
    "rank_colsample": 0.75,
    "rank_l1": 0.5,
    "rank_l2": 10.0,
    "rank_max_bin": 127,
    "truncation_level": 8
  },
  "python": "3.12.13",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "files": {
    "modules_e5.npy": {
      "bytes": 52352,
      "sha256": "bac1185682b58e2fa92a217ff3da2b4d1a59febc7f191ace94445678bfc9b04d"
    },
    "modules_nomic.npy": {


## Output contract

The notebook writes all diagnostics under `/kaggle/working/v136/`. A new `candidate_v136_GC_OWLF.csv` is written **only** when the frozen whole-list gates pass; otherwise the terminal result is `HOLD` and no experimental submission file is produced. This prevents accidental leaderboard use of an unvalidated full-list reranker.